# CrimeScope ML — 02. Ingest & Tract Pipeline (Lakehouse)

**Description:** Crime ingest (Socrata → Delta), tract boundaries (TIGER → Delta), spatial join, monthly tract counts, Delta optimization, and data-quality assertions — all on Unity Catalog.

**Lakehouse features used:**
- Unity Catalog (`varanasi.default.*`)
- Delta Lake with `OPTIMIZE` + `ZORDER`
- Unity Catalog Volumes for raw file staging
- Data-quality assertions before downstream use

**Tables written:**
- `chicago_crimes_raw`
- `chicago_crime_monthly_by_community_area`
- `cook_tract_boundaries`
- `chicago_crimes_with_tract`
- `chicago_crime_monthly_by_tract`

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")
display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema"))

---
## 1. Create Unity Catalog Volume for raw staging

In [ ]:
spark.sql("""
  CREATE VOLUME IF NOT EXISTS varanasi.default.ml_data
  COMMENT 'Raw data staging for CrimeScope ML pipeline'
""")
print("Volume varanasi.default.ml_data ready")

---
## 2. Crime ingest (Socrata → Delta)

In [ ]:
import json
import os
import requests
from pyspark.sql.types import (
    DoubleType, StringType, StructField, StructType,
)

CRIME_START_DATE = "2020-01-01"
MAX_ROWS = 1_500_000
PAGE_SIZE = 50_000
RAW_TABLE = "varanasi.default.chicago_crimes_raw"

APP_TOKEN = os.environ.get("SOCRATA_APP_TOKEN", None)

base_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
all_rows = []
offset = 0

while len(all_rows) < MAX_ROWS:
    params = {
        "$where": f"date >= '{CRIME_START_DATE}'",
        "$limit": PAGE_SIZE,
        "$offset": offset,
        "$order": "date ASC",
    }
    headers = {}
    if APP_TOKEN:
        headers["X-App-Token"] = APP_TOKEN

    resp = requests.get(base_url, params=params, headers=headers, timeout=120)
    resp.raise_for_status()
    page = resp.json()

    if not page:
        break

    all_rows.extend(page)
    offset += len(page)
    print(f"  Fetched {len(all_rows):,} rows so far (page {offset // PAGE_SIZE})")

    if len(page) < PAGE_SIZE:
        break

print(f"\nTotal rows fetched: {len(all_rows):,}")

In [ ]:
schema = StructType([
    StructField("id", StringType()),
    StructField("case_number", StringType()),
    StructField("date", StringType()),
    StructField("block", StringType()),
    StructField("iucr", StringType()),
    StructField("primary_type", StringType()),
    StructField("description", StringType()),
    StructField("location_description", StringType()),
    StructField("arrest", StringType()),
    StructField("domestic", StringType()),
    StructField("beat", StringType()),
    StructField("district", StringType()),
    StructField("ward", StringType()),
    StructField("community_area", StringType()),
    StructField("fbi_code", StringType()),
    StructField("x_coordinate", StringType()),
    StructField("y_coordinate", StringType()),
    StructField("year", StringType()),
    StructField("updated_on", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

import pandas as pd

pdf = pd.DataFrame(all_rows)
schema_cols = [f.name for f in schema.fields]
for col in schema_cols:
    if col not in pdf.columns:
        pdf[col] = None
pdf = pdf[schema_cols]

for field in schema.fields:
    if isinstance(field.dataType, StringType):
        pdf[field.name] = pdf[field.name].astype(object).where(pdf[field.name].notna(), None)
        pdf[field.name] = pdf[field.name].apply(lambda x: str(x) if x is not None else None)
pdf["latitude"] = pd.to_numeric(pdf["latitude"], errors="coerce")
pdf["longitude"] = pd.to_numeric(pdf["longitude"], errors="coerce")

raw_sdf = spark.createDataFrame(pdf, schema=schema)

spark.sql(f"DROP TABLE IF EXISTS {RAW_TABLE}")
raw_sdf.write.saveAsTable(RAW_TABLE)

# Delta Lakehouse: optimize for downstream queries by date and geography
spark.sql(f"OPTIMIZE {RAW_TABLE} ZORDER BY (community_area, date)")

count = spark.table(RAW_TABLE).count()
print(f"Wrote {count:,} rows to {RAW_TABLE} (Delta, Z-ordered by community_area + date)")
display(spark.table(RAW_TABLE).limit(5))

In [ ]:
# Stage a Parquet copy to Unity Catalog Volume for portability
VOLUME_PATH = "/Volumes/varanasi/default/ml_data/chicago_crimes_raw.parquet"
spark.table(RAW_TABLE).write.mode("overwrite").parquet(VOLUME_PATH)
print(f"Staged Parquet copy to {VOLUME_PATH}")

---
## 3. Community-area staging

In [ ]:
from pyspark.sql import functions as F

sdf = spark.table(RAW_TABLE)
monthly_ca = (
    sdf.withColumn("month_start", F.date_trunc("month", F.to_timestamp("date")))
    .groupBy("community_area", "month_start")
    .agg(F.count("*").alias("incident_count"))
    .orderBy("community_area", "month_start")
)

CA_TABLE = "varanasi.default.chicago_crime_monthly_by_community_area"
spark.sql(f"DROP TABLE IF EXISTS {CA_TABLE}")
monthly_ca.write.saveAsTable(CA_TABLE)
spark.sql(f"OPTIMIZE {CA_TABLE}")

print(f"Rows: {spark.table(CA_TABLE).count():,}")
display(spark.table(CA_TABLE).limit(20))

---
## 4. TIGER tract boundaries (Cook County → Delta)

In [ ]:
%pip install geopandas pyogrio -q

In [ ]:
import geopandas as gpd
import os
import tempfile
import urllib.request
import zipfile

url = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_17_tract.zip"
tmp = tempfile.mkdtemp()
zip_path = os.path.join(tmp, "tracts.zip")
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(tmp)

gdf = gpd.read_file(tmp)
cook = gdf[gdf["COUNTYFP"] == "031"].copy()
cook["tract_geoid"] = cook["GEOID"]
cook["wkt"] = cook.geometry.to_wkt()

cols = ["tract_geoid", "STATEFP", "COUNTYFP", "TRACTCE", "NAMELSAD", "ALAND", "AWATER", "wkt"]
pdf = cook[cols].copy()
sdf = spark.createDataFrame(pdf)

TRACT_TABLE = "varanasi.default.cook_tract_boundaries"
spark.sql(f"DROP TABLE IF EXISTS {TRACT_TABLE}")
sdf.write.saveAsTable(TRACT_TABLE)
spark.sql(f"OPTIMIZE {TRACT_TABLE} ZORDER BY (tract_geoid)")

n = spark.table(TRACT_TABLE).count()
print(f"Cook County tracts loaded: {n} (Delta, Z-ordered by tract_geoid)")
display(spark.table(TRACT_TABLE).limit(5))

---
## 5a. Spatial join (crime → tract)

In [ ]:
import geopandas as gpd
from shapely import wkt

crimes_pdf = (
    spark.table(RAW_TABLE)
    .filter("latitude IS NOT NULL AND longitude IS NOT NULL")
    .select(
        "id", "case_number", "date", "primary_type", "description",
        "location_description", "arrest", "domestic", "beat", "district",
        "ward", "community_area", "latitude", "longitude", "updated_on",
    )
    .toPandas()
)

crimes_gdf = gpd.GeoDataFrame(
    crimes_pdf,
    geometry=gpd.points_from_xy(
        crimes_pdf.longitude.astype(float),
        crimes_pdf.latitude.astype(float),
    ),
    crs="EPSG:4326",
)

tracts_pdf = spark.table(TRACT_TABLE).toPandas()
tracts_gdf = gpd.GeoDataFrame(
    tracts_pdf,
    geometry=tracts_pdf["wkt"].apply(wkt.loads),
    crs="EPSG:4326",
)

joined = gpd.sjoin(
    crimes_gdf,
    tracts_gdf[["tract_geoid", "geometry"]],
    how="left",
    predicate="within",
)
joined = joined.drop(columns=["geometry", "index_right"], errors="ignore")

CWT_TABLE = "varanasi.default.chicago_crimes_with_tract"
sdf = spark.createDataFrame(joined)
spark.sql(f"DROP TABLE IF EXISTS {CWT_TABLE}")
sdf.write.saveAsTable(CWT_TABLE)
spark.sql(f"OPTIMIZE {CWT_TABLE} ZORDER BY (tract_geoid, date)")

n = spark.table(CWT_TABLE).count()
print(f"Crimes with tract assignment: {n:,} (Delta, Z-ordered by tract_geoid + date)")

---
## 5b. Monthly tract counts

In [ ]:
from pyspark.sql import functions as F

sdf = spark.table(CWT_TABLE)
monthly = (
    sdf.filter(F.col("tract_geoid").isNotNull())
    .withColumn("month_start", F.date_trunc("month", F.to_timestamp("date")))
    .groupBy("tract_geoid", "month_start")
    .agg(F.count("*").alias("incident_count"))
)

MBT_TABLE = "varanasi.default.chicago_crime_monthly_by_tract"
spark.sql(f"DROP TABLE IF EXISTS {MBT_TABLE}")
monthly.write.saveAsTable(MBT_TABLE)
spark.sql(f"OPTIMIZE {MBT_TABLE} ZORDER BY (tract_geoid, month_start)")

n = spark.table(MBT_TABLE).count()
print(f"Monthly tract rows: {n:,} (Delta, Z-ordered)")
display(spark.table(MBT_TABLE).limit(10))

---
## 6. Data-quality assertions

In [ ]:
from pyspark.sql import functions as F

# --- Raw crime table ---
raw = spark.table(RAW_TABLE)
raw_count = raw.count()
assert raw_count > 100_000, f"FAIL: chicago_crimes_raw has only {raw_count} rows — expected 100k+"

date_range = raw.select(F.min("date").alias("min_d"), F.max("date").alias("max_d")).first()
print(f"✓ chicago_crimes_raw: {raw_count:,} rows, {date_range['min_d']} to {date_range['max_d']}")

# --- Tract boundaries ---
tracts = spark.table(TRACT_TABLE)
n_tracts = tracts.count()
assert n_tracts > 1_300, f"FAIL: cook_tract_boundaries has only {n_tracts} tracts — expected 1300+"
print(f"✓ cook_tract_boundaries: {n_tracts} tracts")

# --- Spatial join coverage ---
cwt = spark.table(CWT_TABLE)
total = cwt.count()
matched = cwt.filter(F.col("tract_geoid").isNotNull()).count()
coverage = matched / max(total, 1) * 100
assert coverage > 90, f"FAIL: spatial join coverage is only {coverage:.1f}% — expected 90%+"
print(f"✓ chicago_crimes_with_tract: {total:,} total, {matched:,} matched ({coverage:.1f}%)")

# --- Monthly tract counts ---
mbt = spark.table(MBT_TABLE)
mbt_stats = mbt.select(
    F.min("month_start").alias("min_month"),
    F.max("month_start").alias("max_month"),
    F.countDistinct("tract_geoid").alias("n_tracts"),
    F.sum("incident_count").alias("total_incidents"),
).first()
print(f"✓ chicago_crime_monthly_by_tract: {mbt.count():,} rows, "
      f"{mbt_stats['n_tracts']} tracts, {mbt_stats['min_month']} to {mbt_stats['max_month']}, "
      f"{mbt_stats['total_incidents']:,} total incidents")

print("\n✅ All data-quality assertions passed — ready for notebook 03.")

---
## 7. Delta table history + metadata

In [ ]:
for table_name in [RAW_TABLE, TRACT_TABLE, CWT_TABLE, MBT_TABLE]:
    print(f"\n=== {table_name} ===")
    display(spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 3"))

---
## 8. Data maturity check (reporting lag)

In [ ]:
display(spark.sql("""
  SELECT 
    date_trunc('month', to_timestamp(date)) AS incident_month,
    COUNT(*) AS incidents,
    ROUND(AVG(datediff(to_date(updated_on), to_date(date))), 1) AS avg_days_to_update,
    MAX(to_date(updated_on)) AS last_update
  FROM varanasi.default.chicago_crimes_raw
  GROUP BY 1 ORDER BY 1 DESC LIMIT 24
"""))

**How to read this table:**

- **`avg_days_to_update`** — average lag between `date` (incident) and `updated_on` (last CPD record update).
- **Rule of thumb:** exclude any month where `avg_days_to_update < 30` from training labels.